# Camada Bronze - Ingestão

## Objetivo
Ingerir os 5 arquivos CSV da Landing Zone e a cotação do dólar obtida via API do BCB,
preservando os dados brutos e adicionando `ingestion_datetime` em cada registro.

**Convenções deste projeto:**
- Comentários e textos: português (PT-BR)
- Identificadores (variáveis, funções): inglês (EN)
  - Constantes: `UPPER_SNAKE_CASE`
  - Demais variáveis: `lower_snake_case`
- Colunas e nomes de tabela: português, `snake_case`
- Símbolos nos logs: ✅ ⚠️ ❌

## Padronização dos nomes
Embora no enunciado fale para dar omitir o catálogo do nome da tabela eu optei por incluir como mostrado em aula para ficar um nome mais robusto.


In [0]:
import json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import to_timestamp, current_timestamp
from datetime import datetime

# Padronização dos nomes dos schemas e volumes
CATALOG = "movies_medallion"
BRONZE_SCHEMA_NAME = "bronze"
SILVER_SCHEMA_NAME = "silver"
GOLD_SCHEMA_NAME = "gold"

BRONZE_SCHEMA = f"{CATALOG}.{BRONZE_SCHEMA_NAME}"
SILVER_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA_NAME}"
GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

LANDING_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA_NAME}/landing"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Para verificação visual
print(f"CATALOG: {CATALOG}")
print(f"BRONZE_SCHEMA: {BRONZE_SCHEMA}")
print(f"SILVER_SCHEMA: {SILVER_SCHEMA}")
print(f"GOLD_SCHEMA: {GOLD_SCHEMA}")
print(f"LANDING_PATH: {LANDING_PATH}")

CATALOG: movies_medallion
BRONZE_SCHEMA: movies_medallion.bronze
SILVER_SCHEMA: movies_medallion.silver
GOLD_SCHEMA: movies_medallion.gold
LANDING_PATH: /Volumes/movies_medallion/bronze/landing


## Ingestão dos dados

Os 5 CSVs são lidos como string e gravados em Delta.

- CSV bruto, sem inferência de schema.
- Coluna `ingestion_datetime` adicionada em cada linha
- Modo `overwrite` em vez de `append`: batelada única, evita duplicatas em reexecução

In [0]:
# Mapeamento explícito arquivo -> nome de tabela, conforme §1.2 do enunciado.
# Não derivamos o nome por manipulação de string (replace/removesuffix) porque
# o mapeamento é uma decisão do PDF, não um padrão inferível.
expected_files = {
    "movies_info_IMDB_TMDB.csv": "movies_info",
    "movies_financials_IMDB_TMDB.csv": "movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "credits_and_tags",
    "movies_reviews.csv": "movies_reviews",
}

for file, name in expected_files.items():
    table = f"{BRONZE_SCHEMA}.tb_{name}"
    file_path = f"{LANDING_PATH}/{file}"

    # Leitura: CSV bruto, sem inferência de schema.
    #    Todas as colunas entram como string — a tipagem é responsabilidade do Silver.
    df = spark.read.csv(file_path, header=True).withColumn(
        "ingestion_datetime", current_timestamp()
    )

    # Escrita: Delta, modo overwrite (batelada única, sem ingestão incremental)
    #    e overwriteSchema para acomodar variações de tipo entre reexecuções.
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )

    # Validação: tabela vazia é erro — mesma política fail-fast do notebook 00.
    count = spark.table(table).count()
    if count == 0:
        print(f"{FAIL} {table} foi gravada vazia")
        raise Exception(
            f"Tabela {table} vazia. Verifique o arquivo {file} na landing zone."
        )
    else:
        print(f"{OK} {table}: {count} linhas")

    # Inspeção visual: amostra das 5 primeiras linhas para conferência rápida.
    display(spark.table(table).limit(5))

✅ movies_medallion.bronze.tb_movies_info: 106930 linhas


id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-16T17:46:58.938Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-16T17:46:58.938Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-16T17:46:58.938Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-16T17:46:58.938Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-16T17:46:58.938Z


✅ movies_medallion.bronze.tb_movies_financials: 106165 linhas


id,budget,revenue,ingestion_datetime
293660,58000000,Unknown,2026-10-16T17:47:05.330Z
299536,300000000,2052415039,2026-10-16T17:47:05.330Z
299534,356000000,2800000000,2026-10-16T17:47:05.330Z
475557,55000000,1074458282,2026-10-16T17:47:05.330Z
271110,250000000,Não Informado,2026-10-16T17:47:05.330Z


✅ movies_medallion.bronze.tb_movies_metrics: 107364 linhas


id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-10-16T17:47:10.006Z
299536,"154,34",8.255,27713,8.4,1406782,2026-10-16T17:47:10.006Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-16T17:47:10.006Z
475557,"54,522",8.168,23425,8.3,1723035,2026-10-16T17:47:10.006Z
271110,70.741,7.4,21541,7.8,947222,2026-10-16T17:47:10.006Z


✅ movies_medallion.bronze.tb_credits_and_tags: 106320 linhas


id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-16T17:47:14.732Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-16T17:47:14.732Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-10-16T17:47:14.732Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-10-16T17:47:14.732Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-10-16T17:47:14.732Z


✅ movies_medallion.bronze.tb_movies_reviews: 32412 linhas


id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-10-16T17:47:19.700Z
637007,Lucas Reis 602,3.9,null,2026-10-16T17:47:19.700Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-16T17:47:19.700Z
413036,Gabriela Monteiro 401,7.5,null,2026-10-16T17:47:19.700Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-16T17:47:19.700Z


## Ingestão da cotação do dólar via API

O Databricks não tem saída para `olinda.bcb.gov.br` neste ambiente, então o JSON foi obtido externamente e colocado na landing zone.

- Endpoint: `CotacaoDolarPeriodo` da API PTAX/BCB
- Período consultado: 29/09/2026 a 06/10/2026
- Retorno: 6 cotações (sábados e domingos sem cotação, como esperado)
- Arquivo na landing: `cotacao_dolar.json`
- Tratamento do JSON feito no Databricks, seguindo o padrão dos outros blocos


In [0]:
# Caminho do arquivo JSON obtido da API do BCB
json_path = f"{LANDING_PATH}/cotacao_dolar.json"

# Lê o JSON
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)
    
cotacao_dolar = data.get("value", [])

# Verifica se o retorno não foi vazio já que o restante do pipeline depende da cotação.
if len(cotacao_dolar) == 0:
    print(f"{FAIL} Nenhuma cotação retornada pela API")
    raise Exception("API do BCB não retornou cotações para o período consultado.")

print(f"{OK} Cotações carregadas: {len(cotacao_dolar)}")
print(f"Exemplo do primeiro registro: {cotacao_dolar[0]}")

# Schema explícito: a API entrega cotacaoCompra como número
# e dataHoraCotacao como string. Aqui já fixamos os tipos esperados.
schema = StructType(
    [
        StructField("cotacaoCompra", DoubleType(), True),
        StructField("dataHoraCotacao", StringType(), True),
    ]
)

df_cotacao = spark.createDataFrame(cotacao_dolar, schema=schema)

# Converte dataHoraCotacao (string) para timestamp
# e adiciona o carimbo de ingestão do Bronze
df_cotacao = df_cotacao.withColumn(
    "dataHoraCotacao", to_timestamp("dataHoraCotacao")
).withColumn("ingestion_datetime", current_timestamp())

df_cotacao.printSchema()
df_cotacao.show(truncate=False)

table = f"{BRONZE_SCHEMA}.tb_cotacao_dolar"

(
    df_cotacao.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table)
)

print(f"{OK} Gravado em: {table}")
display(spark.table(table).orderBy("dataHoraCotacao"))

✅ Cotações carregadas: 6
Exemplo do primeiro registro: {'cotacaoCompra': 5.2198, 'dataHoraCotacao': '2026-09-29 13:06:06.446305'}
root
 |-- cotacaoCompra: double (nullable = true)
 |-- dataHoraCotacao: timestamp (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = false)

+-------------+--------------------------+--------------------------+
|cotacaoCompra|dataHoraCotacao           |ingestion_datetime        |
+-------------+--------------------------+--------------------------+
|5.2198       |2026-09-29 13:06:06.446305|2026-10-16 17:47:24.161088|
|5.1803       |2026-09-30 13:11:44.783262|2026-10-16 17:47:24.161088|
|5.2073       |2026-10-01 13:10:35.4469  |2026-10-16 17:47:24.161088|
|5.2232       |2026-10-02 13:03:16.256632|2026-10-16 17:47:24.161088|
|4.9853       |2026-10-05 13:07:36.558602|2026-10-16 17:47:24.161088|
|4.9692       |2026-10-06 13:03:21.267287|2026-10-16 17:47:24.161088|
+-------------+--------------------------+--------------------------+

✅ Gravado em: 

cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.2198,2026-09-29T13:06:06.446Z,2026-10-16T17:47:25.336Z
5.1803,2026-09-30T13:11:44.783Z,2026-10-16T17:47:25.336Z
5.2073,2026-10-01T13:10:35.446Z,2026-10-16T17:47:25.336Z
5.2232,2026-10-02T13:03:16.256Z,2026-10-16T17:47:25.336Z
4.9853,2026-10-05T13:07:36.558Z,2026-10-16T17:47:25.336Z
4.9692,2026-10-06T13:03:21.267Z,2026-10-16T17:47:25.336Z


## Resumo Final — Ingestão Bronze

Apenas reporta. Validações por bloco já deram raise antes.

In [0]:
# Resumo final: apenas reporta o estado após as validações acima.
tables = [
    "tb_movies_info",
    "tb_movies_financials",
    "tb_movies_metrics",
    "tb_credits_and_tags",
    "tb_movies_reviews",
    "tb_cotacao_dolar",
]

print("=" * 60)
print("RESUMO DA INGESTÃO BRONZE")
print("=" * 60)

print(f"Tabelas em {BRONZE_SCHEMA}:\n")
display(spark.sql(f"SHOW TABLES IN {BRONZE_SCHEMA}"))

for t in tables:
    full = f"{BRONZE_SCHEMA}.{t}"
    n = spark.table(full).count()
    print(f"{OK} {full}: {n} linhas")

print(f"Executado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

RESUMO DA INGESTÃO BRONZE
Tabelas em movies_medallion.bronze:



database,tableName,isTemporary
bronze,tb_cotacao_dolar,false
bronze,tb_credits_and_tags,false
bronze,tb_movies_financials,false
bronze,tb_movies_info,false
bronze,tb_movies_metrics,false
bronze,tb_movies_reviews,false


✅ movies_medallion.bronze.tb_movies_info: 106930 linhas
✅ movies_medallion.bronze.tb_movies_financials: 106165 linhas
✅ movies_medallion.bronze.tb_movies_metrics: 107364 linhas
✅ movies_medallion.bronze.tb_credits_and_tags: 106320 linhas
✅ movies_medallion.bronze.tb_movies_reviews: 32412 linhas
✅ movies_medallion.bronze.tb_cotacao_dolar: 6 linhas
Executado em: 2026-10-16 17:47:30
